# Creating Vanilla Rainbow Tables

For N = 2**16, currently only making one table

## Imports

In [1]:
import pickle
import random
from hashlib import sha256
from tqdm import tqdm
from math import pi, sqrt, e, log

## Table Parameters

In [2]:
# initialise startpoints - either generate them or load from pickle - to keep same across runs
def get_startpoints(N, m_0, nlabel, alpha):
    # try opening pickle file, else generate and save
    try:
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'rb') as f:
            startpoints = pickle.load(f)

    # no file found - generate and save
    except FileNotFoundError:
        # random but unique - store as a set?
        startpoints = set()
        while len(startpoints) < m_0:
            startpoints.add(random.randint(0, N-1))

        # store startpoints in pickle file
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'wb') as f:
            pickle.dump(startpoints, f)

    # return the startpoints
    return startpoints

In [3]:
# label N to find easier - label is the exponent
nlabel = 16
##############################################################################
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%
##############################################################################
t = round(log(1-p)/log(1-N**(-1/3))) # chain length t
alpha = 0.95 # maximality factor
mt_target = N**(2/3) # our target mt
m_0 = round(mt_target/(1-alpha))    # m_0 - number of startpoints
##############################################################################
# initialise startpoints 
startpoints = get_startpoints(N, m_0, nlabel, alpha)


## Hash and Reduction Functions

In [4]:
# Hash function
def H(x):
	return int(sha256(bytes(x)).hexdigest(), 16)

# Reduction function
# currently mod but should change to murmurhash in future
def r(y, i, ell=0):   # also takes in ell - number of tables - for future use (but currently ell=0)
	return (y + i + ell*t) % N

## Building Vanilla Table

In [5]:
# take in m_0 and t as parameters - how many chains to start with and how long to make the chains
# store the table as a dictionary of endpoint:startpoint pairs (rather than sp:ep for easier lookup later)
# store in a pickle file

def build_vanilla_table(m_0, t, alpha, startpoints):
    # store table in dictionary
    table = {}

    # for each startpoint
    for j in tqdm(range(m_0)):
        # pop the next startpoint off
        sp = startpoints.pop()
        current_point = sp  # keep track of current point in chain -  we want to store startpoint later

        # create the chain
        for i in range(t):
            # hash then reduce the value
            current_point = r(H(current_point), i)

        # check if there wasn't a chain merge (not in a value stored already) - if not then store in table
        if current_point not in table.keys():
            table[current_point] = sp

    # store the table as a pickle file
    with open(f'vanilla_table_alpha_{alpha}_t_{t}.pkl', 'wb') as f:
        pickle.dump(table, f)

    return table

## Searching the Table

In [6]:
def search_vanilla_table(y, t, table):
    # keep track of hashes and reductions
    hashes = 0
    reductions = 0

    # keep track of column we're in 
    c = t - 1
    # reduce (r_t-1) the hash then compare in the table
    x = r(y, c)
    reductions += 1

    # while we haven't reached the end of our chain
    while c > 0:
        # if there is a match in the keys (our endpoints), regenerate chain until we find the key
        try:
            point = table[x]   # search for endpoint in table and get startpoint
            # regenerate chain until column c - we are now in the column before the match
            for i in range(c):
                point = r(H(point), i)
                hashes += 1
                reductions += 1
            
            # hash the point - if it is a match we have found our preimage 
            if H(point) == y:
                hashes += 1
                return point, hashes, reductions
            
            # do we need this line????
            else:
                return "false alarm", hashes, reductions
            
            # if that didn't work then we ran into a false alarm
 
        # if we didn't find a match in endpoints, we need to restart the search
        except:
            # reduce c by 1 to move to the previous column
            c -= 1
            # number of columns between current column and end 
            # diff = t - c  
            # reduce y by the new c index
            x = r(y, c)
            reductions += 1
            # then hash and reduce however many times to move back through columns
            for i in range((t - c) - 1, 0, -1):    # decrease difference by 1 (as we already reduced by current diff index) then continuously reduce by 1
                x = r(H(x), t-i)
                hashes += 1
                reductions += 1

    # We have searched all columns - return -1
    return -1, hashes, reductions

## Run

### Precomputation Phase - Build the Table

In [7]:
# either build or load table
def get_vanilla_table():
    # try loading table from pickle file
    try:
        with open(f'vanilla_table_alpha_{alpha}_t_{t}.pkl', 'rb') as f:
            table = pickle.load(f)

    # if no pickle file found, build the table
    except FileNotFoundError:
        table = build_vanilla_table(m_0, t, alpha, startpoints)

    return table

In [8]:
vanilla = get_vanilla_table()

### Online Phase - Using the Table

In [9]:
# get chain from pickle file
with open(f'vanilla_chain_alpha_{alpha}_t_{t}.pkl', 'rb') as f:
    chain = pickle.load(f)

# test each column in the chains
for i in range(1, len(chain)):
    print(f"Column {t - i}:")
    print(f"     key = {chain[-(1+i)]}")
    y = H(chain[-(1+i)])
    value, hashes, reductions = search_vanilla_table(y, t, vanilla)
    print(f"     {value}")
    print(f'     Hashes: {hashes}, Reductions: {reductions}')

Column 79:
     key = 55574


     55574
     Hashes: 80, Reductions: 80
Column 78:
     key = 64392
     64392
     Hashes: 80, Reductions: 81
Column 77:
     key = 31136
     31136
     Hashes: 81, Reductions: 83
Column 76:
     key = 13361
     13361
     Hashes: 83, Reductions: 86
Column 75:
     key = 26168
     26168
     Hashes: 86, Reductions: 90
Column 74:
     key = 16752
     false alarm
     Hashes: 79, Reductions: 80
Column 73:
     key = 2730
     2730
     Hashes: 95, Reductions: 101
Column 72:
     key = 16996
     16996
     Hashes: 83, Reductions: 86
Column 71:
     key = 5050
     5050
     Hashes: 108, Reductions: 116
Column 70:
     key = 44219
     false alarm
     Hashes: 100, Reductions: 108
Column 69:
     key = 65490
     65490
     Hashes: 125, Reductions: 135
Column 68:
     key = 51880
     51880
     Hashes: 81, Reductions: 83
Column 67:
     key = 58400
     false alarm
     Hashes: 79, Reductions: 81
Column 66:
     key = 21488
     false alarm
     Hashes: 145, Reductions: 158
Colum